In [1]:
import requests
import pandas as pd
import time
from datetime import date
from pathlib import Path

In [2]:
import pandas as pd

URL = ("https://raw.githubusercontent.com/remy000/"
       "Rwanda-price-tracker/main/data/price_history.csv")

In [3]:
import time
df = pd.read_csv(URL + f"?t={int(time.time())}",
                 dtype={"product_id": "int64"})

df.category.value_counts().head(20)

category
uncategorised          9951
beauty                   74
Light Bulbs              68
Fruits & Vegetables      51
BEAUTY                   28
Skin Care                23
drinks                   22
perfume                  20
Food                     19
Liquor                   19
Kitchen materials        18
ELECTRICAL PRODUCT       17
Juice                    16
Toys                     15
soap                     15
Biscuit                  14
cake handle              12
hair                     12
Liquid soap              12
SOAP                     12
Name: count, dtype: int64

In [4]:
staples = ["rice", "sugar", "oil", "salt", "flour", "milk", "soap",
           "beans", "maize", "bread", "egg", "tea", "water"]

for s in staples:
    loose = df.title.str.contains(s, case=False, na=False).sum()
    tight = df.title.str.contains(rf"\b{s}\b", case=False, na=False).sum()
    print(f"{s:8} {loose:5} -> {tight:5}")

rice        97 ->    84
sugar       75 ->    69
oil        448 ->   322
salt        75 ->    51
flour       79 ->    78
milk       173 ->   166
soap       189 ->   187
beans       39 ->    39
maize       16 ->    16
bread       36 ->    23
egg         46 ->    24
tea        259 ->   154
water      205 ->   170


In [5]:
JUNK = ("cookie|cracker|snack|spoon|paddle|seasoning|spice|cake|toy|"
        "dish|holder|machine|cooker|maker|iron|bulb|"
        "sugar.?free|bowl|dispenser|candy|candies|chewing|gum|biscuit|"
        "chocolate|cola|spirit|icing|caster|zero")

def show(term, n=25):
    hits = df[df.title.str.contains(rf"\b{term}\b", case=False, na=False)]
    hits = hits[~hits.title.str.contains(JUNK, case=False, na=False)]
    out = hits[["shop", "product_id", "title", "price_rwf"]].copy()
    out["title"] = out.title.str.slice(0, 50)
    return out.reset_index(drop=True).head(n)

show("sugar")

,shop,product_id,title,price_rwf
0,amahaho,7235416162475,Brown Sugar,2750.0
1,amahaho,7253653946539,Brown Sugar - Isukari,5800.0
2,amahaho,7866171195563,Brown Sugar ILLOVO / 1Kg,2300.0
3,amahaho,7499982864555,Everyday - Sugar Cubes - Sucre en Morceaux /Kg,9500.0
4,murukali,9358756085991,Azam BGA Moyo Sugar 1kg,3000.0
5,murukali,4652234014859,Brown sugar /kg,2100.0
6,murukali,8512139395303,CANDICO CANE SUGAR LUMPS FT 1KG,15000.0
7,murukali,9524709163239,Candico Bio Brown Cane Sugar Cubes (1kg),18000.0
8,murukali,9464692310247,Cane Sugar Brown Demerara 100% Natural 500g,9000.0
9,murukali,9133427130599,Daily Flesh Granulated Sugar 2kg,12000.0


In [6]:
for term in ["salt", "cooking oil", "milk", "flour", "beans",
             "maize", "bread", "soap", "egg", "tea"]:
    print(f"\n===== {term.upper()} =====")
    print(show(term, 15).to_string(index=False))


===== SALT =====
    shop    product_id                                             title  price_rwf
 amahaho 7888421585067         Cooking Salt/ Sel de cuisine/ Umunyu /1kg      710.0
 amahaho 8477545562283                           Lay’s Edible Salt Chips     2800.0
 amahaho 8477514891435             Lay’s Salt and Vinegar Flavored Chips      600.0
 amahaho 7599431614635                                         Nezo Salt     5000.0
 amahaho 8477624041643                      Winnaz Salt & Vinegar Crisps      950.0
murukali 9223783416039                         Coffee Sea Salt Bath 350g     9400.0
murukali 9554317738215     Coffee Sea Salt Bath Whitening SPA Scrub/680g    11500.0
murukali 9296252731623                         Everyday Cooking Salt 1kg     3600.0
murukali 9132594331879       Forever Truly Tasty Rock Salt  Sea Salt 1kg    12000.0
murukali 8636839919847                       HEMANI Salt Oil 30mL (1 OZ)     5000.0
murukali 9082992296167         HIMALAYAN SALT Body Scrub F

In [7]:
import re

STAPLES = {
    "rice":          ["rice", "umuceli"],
    "maize flour":   ["maize flour", "kawunga", "maize meal"],
    "wheat flour":   ["wheat flour", "baking flour"],
    "cassava flour": ["cassava flour", "cassava"],
    "sorghum":       ["sorghum", "millet", "amasaka"],
    "pasta":         ["spaghetti", "macaroni", "pasta", "noodles"],
    "bread":         ["bread"],
    "beans":         ["beans", "ibishyimbo", "peas", "amashaza"],
    "groundnut":     ["groundnut", "peanut", "ubunyobwa"],
    "egg":           ["egg"],
    "meat":          ["beef", "goat meat", "pork", "chicken", "inyama"],
    "fish":          ["fish", "sambaza", "tilapia"],
    "milk":          ["milk", "amata"],
    "yoghurt":       ["yoghurt", "yogurt", "ikivuguto"],
    "cooking oil":   ["cooking oil", "palm olein", "vegetable oil", "huile"],
    "margarine":     ["margarine", "blueband", "blue band"],
    "sugar":         ["sugar", "isukari"],
    "salt":          ["cooking salt", "table salt", "umunyu"],
    "tea":           ["tea bags", "black tea", "icyayi"],
    "coffee":        ["coffee", "ikawa"],
    "potato":        ["potato", "ibirayi"],
    "onion":         ["onion", "ibitunguru"],
    "tomato":        ["tomato", "inyanya"],
    "cabbage":       ["cabbage", "amashu"],
    "carrot":        ["carrot", "karoti"],
    "banana":        ["banana", "plantain", "igitoke"],
    "avocado":       ["avocado", "avoka"],
    "fruit":         ["pineapple", "mango", "orange", "passion fruit"],
    "soap":          ["bar soap", "washing soap", "laundry soap", "savon"],
    "detergent":     ["detergent", "omo", "washing powder"],
    "toilet paper":  ["toilet paper", "tissue"],
    "toothpaste":    ["toothpaste", "colgate"],
    "water":         ["drinking water", "mineral water", "still water"],
}

JUNK_WORDS = [
    "cookie", "cracker", "snack", "crisp", "crisps", "chips", "biscuit",
    "candy", "candies", "chewing", "gum", "chocolate", "wafer",
    "spoon", "paddle", "bowl", "dispenser", "basket", "holder", "machine",
    "cooker", "maker", "toaster", "iron", "kettle", "flask", "pot", "pan",
    "cutter", "slicer", "beater", "timer", "boiler", "grinder", "blender",
    "tray", "jar", "mug", "cup", "plate", "knife", "board", "sifter",
    "peeler", "utensil",
    "shampoo", "lotion", "scrub", "cream", "serum", "whitening", "facial",
    "face wash", "body", "hair", "perfume", "deodorant", "bath", "shower",
    "glutathione", "kojic", "collagen", "spa", "mask", "gel", "balm",
    "clay", "shea", "petroleum", "conditioner", "cleanser", "toner",
    "wellness", "therapy", "essential oil", "avocado oil", "body butter",
    "himalayan", "slimming", "supplement", "vitamin", "infant",
    "formula", "gluten", "vegan", "zero", "refined",
    "beer", "wine", "whisky", "vodka", "gin", "spirit", "liqueur",
    "cola", "soda",
]

JUNK = "|".join(rf"\b{w}\b" for w in JUNK_WORDS)
JUNK += r"|sugar.?free|fat burner|energy drink|x\s?\d{2}|\d{2}\s?x"

SIZE = re.compile(
    r"(?:\d+\s*\.?\d*\s*(?:kg|kgs|g|gr|grams?|l|lt|ltr|litres?|ml|cl|pcs?|pieces?)\b)"
    r"|(?:/\s*(?:kg|g|l|ml|pc|pcs|piece)\b)",
    re.IGNORECASE,
)

In [8]:
PER_STAPLE = 14

def find_candidates(data):
    latest = data[data.date == data.date.max()].copy()
    print(f"{len(latest):,} products in catalogue\n")

    latest = latest[~latest.title.str.contains(JUNK, case=False, na=False, regex=True)]
    latest = latest[latest.title.str.contains(SIZE, na=False)]

    picks = []
    for staple, terms in STAPLES.items():
        pattern = "|".join(rf"\b{re.escape(t)}(?:s|es)?\b" for t in terms)
        hits = latest[latest.title.str.contains(pattern, case=False, na=False)].copy()

        if hits.empty:
            print(f"  {staple:15} none found")
            continue

        hits["staple"] = staple
        hits = hits.sort_values("price_rwf")
        step = max(1, len(hits) // PER_STAPLE)
        hits = hits.iloc[::step].head(PER_STAPLE)

        print(f"  {staple:15} {len(hits):3}")
        picks.append(hits)

    return pd.concat(picks, ignore_index=True)

In [10]:
cands = find_candidates(df)
print(f"\n{len(cands)} candidates")
out = (cands[["staple", "shop", "product_id", "title", "price_rwf"]]
       .sort_values(["staple", "price_rwf"]))

out.to_csv("basket_candidates.csv", index=False)
print(f"{len(out)} rows written")

11,588 products in catalogue

  rice             14
  maize flour      13
  wheat flour       9
  cassava flour    11
  sorghum           9
  pasta            14
  bread             2
  beans            14
  groundnut        14
  egg               9
  meat             14
  fish             11
  milk             14
  yoghurt          14
  cooking oil      14
  margarine        14
  sugar            14
  salt              2
  tea              14
  coffee           14
  potato            7
  onion             7
  tomato           14
  cabbage           4
  carrot           11
  banana            8
  avocado           1
  fruit            14
  soap             14
  detergent        14
  toilet paper      8
  toothpaste       14
  water            14

364 candidates
364 rows written


In [12]:
basket = pd.read_excel("basket.xlsx", dtype={"product_id": "int64"})
print(f"basket defines {len(basket)} items")

tracked = df.merge(basket[["product_id", "staple"]], on="product_id")

found = tracked.product_id.nunique()
print(f"found {found} of {len(basket)} in today's scrape")

missing = set(basket.product_id) - set(tracked.product_id)
if missing:
    print("\nMISSING:")
    print(basket[basket.product_id.isin(missing)][["shop", "title"]].to_string(index=False))

basket defines 148 items
found 148 of 148 in today's scrape


In [13]:
daily = tracked.groupby("date").agg(
    items=("product_id", "nunique"),
    basket_cost=("price_rwf", "sum"),
)
daily

,items,basket_cost
date,,
2026-07-28,148,633995.0
